# Lab 45: Anchoring the consensus

[Lab 43](../43-annotator-drift/) ranked annotators against the majority consensus - which is circular, since the consensus is built from them. Add a gold label, show the majority can be collectively wrong, and re-rank annotators against gold. The evaluation-side twin of [Lab 44](../44-hardening-the-signals/)'s held-out baseline. Fill in the `TODO` cells; reference in `solution/`.

## Step 0: Setup

In [ ]:
import json
import pathlib
from sklearn.metrics import cohen_kappa_score, accuracy_score
print("anchoring the consensus to a gold set")

## Step 1: Annotations plus a gold label

In [ ]:
# Lab 43 ranked annotators by agreement with the majority CONSENSUS. But the consensus is
# built from those same annotators - if they are collectively wrong, it is wrong, and
# ranking against it is circular. Here each item also has a GOLD label: an expert /
# adjudicated answer treated as ground truth.
with open("./annotations_with_gold.jsonl") as f:
    items = [json.loads(line) for line in f]
consensus=[1 if (it["a1"]+it["a2"]+it["a3"])>=2 else 0 for it in items]
gold=[it["gold"] for it in items]
print(f"{len(items)} items, 3 annotators + a gold label")

## Step 2: Is the majority actually right?

Consensus vs gold.

In [ ]:
# TODO: compute accuracy_score(gold, consensus) and list the item ids where consensus !=
# gold (the items the majority got collectively wrong). Confirm accuracy < 1.0.
raise NotImplementedError

## Step 3: Consensus-ranking misranks annotators

In [ ]:
# The consequence: an annotator's agreement-with-consensus can misrank them. Compare each
# annotator against consensus AND against gold.
print(f"{'ann':<4}{'vs-consensus':>14}{'vs-gold':>10}")
for a in ["a1","a2","a3"]:
    kc=cohen_kappa_score([it[a] for it in items], consensus)
    kg=cohen_kappa_score([it[a] for it in items], gold)
    print(f"{a:<4}{kc:>14.2f}{kg:>10.2f}")
print("\na3 looks worst against consensus but recovers against gold - on the items the")
print("majority got wrong, a3 was right. Consensus-only ranking underrated it.")

## Step 4: Anchor the weights to gold

In [ ]:
# TODO: build consensus-anchored and gold-anchored weight dicts (cohen_kappa_score of each
# annotator vs consensus, and vs gold). Print both rankings and show they can differ.
raise NotImplementedError

## Step 5: Spend adjudication where it matters

In [ ]:
# You cannot afford gold on everything. Spend it where it matters: items where annotators
# disagree (adjudication queue) and a sample of where they all agree (catch collective error).
adjudicate=[it["id"] for it in items if len({it["a1"],it["a2"],it["a3"]})>1]
unanimous=[it["id"] for it in items if len({it["a1"],it["a2"],it["a3"]})==1]
print(f"adjudication queue (annotators split): {len(adjudicate)} items")
print(f"unanimous items (sample a few for gold, to catch collective error): {len(unanimous)}")
print("Gold-label the disagreements in full; spot-check the agreements - that is where a")
print("confident, wrong consensus hides.")

## Step 6: The same move as the held-out baseline

In [ ]:
# This is the same move Lab 44 made for the drift baseline: do not measure against
# yourself. The drift baseline measured on the model's own training data is optimistic;
# the judge measured against a consensus built from the same annotators is circular. Both
# are fixed by an EXTERNAL anchor - a held-out clean sample there, a gold set here.
print("Held-out reference : drift baseline :: gold set : annotator consensus.")
print("Anchor every reference to something you did not derive from the thing being judged.")

## Step 7: The discipline

In [ ]:
# The discipline: treat the consensus as a hypothesis, not the truth. Anchor it with gold
# on the items that matter, rank annotators by agreement with gold, and re-check the
# judge's ceiling (Lab 40) against gold - not against a vote that can be confidently wrong.
print("A majority can be confidently wrong. Anchor to gold, or you will calibrate your")
print("judge and your annotators against your own mistakes.")

## What you built

A gold-anchored view of annotation quality: consensus-vs-gold accuracy (the majority is collectively wrong on some items, so consensus is not ground truth), per-annotator agreement against gold vs against consensus (which can rank annotators differently and rescue one the consensus underrated), gold-anchored reliability weights, and an adjudication strategy that gold-labels disagreements in full and spot-checks agreements to catch confident collective error.

**Where this simplifies:** gold itself isn't free of error (use multiple experts / adjudication for the gold too); twenty items is a teaching size; and the κ values swing on small samples, so treat the ranking shift as directional, not precise. The key idea is structural, not numerical: anchor every reference to something you did not derive from the thing being judged.

This closes the evaluation-quality thread that ran from [Lab 40](../40-annotation-quality/) (the ceiling) through [Lab 43](../43-annotator-drift/) (annotators drift) to here (the consensus they vote on is itself only a hypothesis until gold anchors it).